In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit

# Define the API endpoint URL
url = "https://api.nhtsa.gov/products/vehicle/modelYears?issueType=c"

# Make a GET request to the API
resp = requests.get(url)
resp.raise_for_status()
data = resp.json()

# Extract model years from the API response and create Row objects
rows = [
    Row(model_year=int(y["modelYear"]))
    for y in data.get("results", [])
    if y.get("modelYear") is not None
]

# Create a Spark DataFrame and add ingestion timestamp and source columns
df_years = spark.createDataFrame(rows) \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source", lit(url))


In [0]:
df_years.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.model_years")